# Selective one-bit weight research

Read-only component study on the prepared public Qwen checkpoint. No task, harness, model checkpoint or other job is modified. No training, admission, speed, peak-memory or 45-second/1359-MiB claim. Tests a sign/scale/exception storage format against native NF4 and FP16 values. All precomputation happens in this diagnostic; no model tensors or caches are exported.


In [ ]:
import base64,hashlib,json,os,runpy,sys,tempfile
from pathlib import Path
assert os.environ.get('SOUP_RESULTS') and os.environ.get('SOUP_REPO')
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
PAYLOAD={'codec.py': 'IiIiRXhwZXJpbWVudGFsIG9uZS1iaXQgc2lnbnMgKyBGUDE2IHNjYWxlICsgZXhhY3QgZXhjZXB0aW9uIHZhbHVlcy4KClRoaXMgaXMgYSBzdG9yYWdlL2Vycm9yIHJlc2VhcmNoIGNvZGVjLCBub3QgYSBjb21wZXRpdGlvbiBzb2x1dGlvbiBvciBhIHNwZWVkCmNsYWltLiBORjQgc2NhbGVzIHJlbWFpbiBzaGFyZWQgd2l0aCB0aGUgcmVmZXJlbmNlIGFuZCBhcmUgZXhjbHVkZWQgb24gYm90aApzaWRlcyBvZiB0aGUgYnl0ZSBjb21wYXJpc29uLiBUaGUgdmFyaWFibGUtcmVjb3JkIGZvcm1hdCBjb3VudHMgbW9kZSBieXRlcywKMzItYml0IG9mZnNldHMsIHNjYWxlcywgYml0bWFwcyBhbmQgZXhhY3QgZXhjZXB0aW9uIHBheWxvYWRzLgoiIiIKaW1wb3J0IG51bXB5IGFzIG5wCgpCTE9DSz02NApORjQ9bnAuYXJyYXkoWy0xLiwtLjY5NjE5MjgwMDk5ODY4NzcsLS41MjUwNzMwNTE0NTI2MzY3LC0uMzk0OTE3NDg4MDk4MTQ0NTMsCiAtLjI4NDQ0MTM4MTY5Mjg4NjM1LC0uMTg0NzczNDMwMjI4MjMzMzQsLS4wOTEwNTAwMzYyNTE1NDQ5NSwwLiwKIC4wNzk1ODAyOTk1NTYyNTUzNCwuMTYwOTMwMjAxNDExMjQ3MjUsLjI0NjExMjMwMTk0NTY4NjM0LC4zMzc5MTUyNDE3MTgyOTIyNCwKIC40NDA3MDk4MjkzMzA0NDQzNCwuNTYyNjE3MDAzOTE3Njk0MSwuNzIyOTU2ODM2MjIzNjAyMywxLl0sZHR5cGU9bnAuZmxvYXQzMikKCmRlZiBuaWJibGVzKHZhbHVlcyk6CiAgICBhPW5wLmFzYXJyYXkodmFsdWVzLGR0eXBlPW5wLnVpbnQ4KS5yZXNoYXBlKC0xKQogICAgaWYgbnAuYW55KGE+MTUpOnJhaXNlIFZhbHVlRXJyb3IoJ05GNCBjb2RlIG91dHNpZGUgMC4uMTUnKQogICAgaWYgYS5zaXplJTI6YT1ucC5jb25jYXRlbmF0ZSgoYSxucC56ZXJvcygxLGR0eXBlPW5wLnVpbnQ4KSkpCiAgICByZXR1cm4gKChhWzo6Ml08PDQpfGFbMTo6Ml0pLnRvYnl0ZXMoKQoKZGVmIHVubmliYmxlcyhyYXcsY291bnQpOgogICAgYT1ucC5mcm9tYnVmZmVyKHJhdyxkdHlwZT1ucC51aW50OCkKICAgIG91dD1ucC5lbXB0eShhLnNpemUqMixkdHlwZT1ucC51aW50OCk7b3V0Wzo6Ml09YT4+NDtvdXRbMTo6Ml09YSYxNQogICAgcmV0dXJuIG91dFs6Y291bnRdCgpkZWYgcmVmZXJlbmNlKHZhbHVlcyxiaXRzLGNvZGVib29rPU5GNCk6CiAgICBhPW5wLmFzYXJyYXkodmFsdWVzKQogICAgaWYgYS5uZGltIT0yIG9yIGEuc2hhcGVbMV0hPUJMT0NLIG9yIGEuc2hhcGVbMF09PTA6cmFpc2UgVmFsdWVFcnJvcignRXhwZWN0ZWQgbm9uZW1wdHkgW2Jsb2Nrcyw2NF0nKQogICAgaWYgYml0cz09NDoKICAgICAgICBpZiBucC5hbnkoYTwwKSBvciBucC5hbnkoYT4xNSkgb3IgbnAuYW55KGEhPWEuYXN0eXBlKG5wLnVpbnQ4KSk6cmFpc2UgVmFsdWVFcnJvcignSW52YWxpZCBORjQgY29kZXMnKQogICAgICAgIHJldHVybiBucC5hc2FycmF5KGNvZGVib29rLGR0eXBlPW5wLmZsb2F0MzIpW2EuYXN0eXBlKG5wLnVpbnQ4KV0KICAgIGlmIGJpdHM9PTE2OgogICAgICAgIGg9YS5hc3R5cGUobnAuZmxvYXQxNikKICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUoaCkuYWxsKCk6cmFpc2UgVmFsdWVFcnJvcignRmluaXRlIEZQMTYgcmVmZXJlbmNlIHJlcXVpcmVkJykKICAgICAgICByZXR1cm4gaC5hc3R5cGUobnAuZmxvYXQzMikKICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1JlZmVyZW5jZSBiaXRzIG11c3QgYmUgNCBvciAxNicpCgpkZWYgZW5jb2RlKHZhbHVlcyxiaXRzLGJ1ZGdldCxjb2RlYm9vaz1ORjQpOgogICAgIiIiQm91bmQgbm9ybWFsaXplZCBibG9jayBSTVMgZXJyb3I7IGV4YWN0IGV4Y2VwdGlvbnMgYW5kIG5hdGl2ZSBmYWxsYmFjay4iIiIKICAgIGlmIG5vdCBucC5pc2Zpbml0ZShidWRnZXQpIG9yIGJ1ZGdldDwwOnJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgZXJyb3IgYnVkZ2V0JykKICAgIG9yaWdpbmFsPW5wLmFzYXJyYXkodmFsdWVzKTt4PXJlZmVyZW5jZShvcmlnaW5hbCxiaXRzLGNvZGVib29rKQogICAgYWxwaGE9bnAubWVhbihucC5hYnMoeCksYXhpcz0xLGR0eXBlPW5wLmZsb2F0MzIpLmFzdHlwZShucC5mbG9hdDE2KQogICAgc2lnbnM9eD49MAogICAgYXBwcm94PW5wLndoZXJlKHNpZ25zLDEuLC0xLikuYXN0eXBlKG5wLmZsb2F0MzIpKmFscGhhLmFzdHlwZShucC5mbG9hdDMyKVs6LE5vbmVdCiAgICBlcnJvcj0oeC5hc3R5cGUobnAuZmxvYXQ2NCktYXBwcm94LmFzdHlwZShucC5mbG9hdDY0KSkqKjIKICAgIGVuZXJneT1ucC5zdW0oeC5hc3R5cGUobnAuZmxvYXQ2NCkqKjIsYXhpcz0xKQogICAgb3JkZXI9bnAuYXJnc29ydCgtZXJyb3IsYXhpcz0xLGtpbmQ9J3N0YWJsZScpCiAgICBzb3J0ZWRfZXJyb3I9bnAudGFrZV9hbG9uZ19heGlzKGVycm9yLG9yZGVyLGF4aXM9MSkKICAgIHJlbWFpbmluZz1ucC5tYXhpbXVtKGVycm9yLnN1bShheGlzPTEpWzosTm9uZV0tbnAuY29uY2F0ZW5hdGUoKG5wLnplcm9zKChsZW4oeCksMSkpLG5wLmN1bXN1bShzb3J0ZWRfZXJyb3IsYXhpcz0xKSksYXhpcz0xKSwwLikKICAgICMgQWxsIDY0IHZhbHVlcyByZXN0b3JlZCBtZWFucyBleGFjdGx5IHplcm8gZXJyb3IsIHJlZ2FyZGxlc3Mgb2YgcmVkdWN0aW9uCiAgICAjIHJvdW5kaW5nIGluIHRoZSBkaWZmZXJlbmNlIG9mIHR3byBpbmRlcGVuZGVudGx5IGFjY3VtdWxhdGVkIEZQNjQgc3Vtcy4KICAgIHJlbWFpbmluZ1s6LC0xXT0wLgogICAgY291bnRzPW5wLmFyZ21heChyZW1haW5pbmc8PSgoYnVkZ2V0KmJ1ZGdldCkqZW5lcmd5KVs6LE5vbmVdKzFlLTMwLGF4aXM9MSkKICAgIHJhdz1ieXRlYXJyYXkoKTtvZmZzZXRzPVswXTttb2Rlcz1bMCwwLDBdO2V4Y2VwdGlvbl9jb3VudD0wCiAgICBmb3IgaSBpbiByYW5nZShsZW4oeCkpOgogICAgICAgIGRlbnNlPW5pYmJsZXMob3JpZ2luYWxbaV0pIGlmIGJpdHM9PTQgZWxzZSB4W2ldLmFzdHlwZSgnPGYyJykudG9ieXRlcygpCiAgICAgICAgaz1pbnQoY291bnRzW2ldKTttYXNrPW5wLnplcm9zKEJMT0NLLGR0eXBlPW5wLmJvb2xfKTttYXNrW29yZGVyW2ksOmtdXT1UcnVlCiAgICAgICAgYmluYXJ5PW5wLnBhY2tiaXRzKHNpZ25zW2ldLGJpdG9yZGVyPSdsaXR0bGUnKS50b2J5dGVzKCkrbnAuYXNhcnJheShhbHBoYVtpXSxkdHlwZT0nPGYyJykudG9ieXRlcygpCiAgICAgICAgaWYgaz09MDpyZWNvcmQ9YidceDAxJytiaW5hcnk7bW9kZT0xCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZXhjZXB0aW9ucz1uaWJibGVzKG9yaWdpbmFsW2ldW21hc2tdKSBpZiBiaXRzPT00IGVsc2UgeFtpXVttYXNrXS5hc3R5cGUoJzxmMicpLnRvYnl0ZXMoKQogICAgICAgICAgICByZWNvcmQ9YidceDAyJytiaW5hcnkrbnAucGFja2JpdHMobWFzayxiaXRvcmRlcj0nbGl0dGxlJykudG9ieXRlcygpK2V4Y2VwdGlvbnM7bW9kZT0yCiAgICAgICAgaWYgbGVuKHJlY29yZCk+PTErbGVuKGRlbnNlKTpyZWNvcmQ9YidceDAwJytkZW5zZTttb2RlPTA7az0wCiAgICAgICAgcmF3LmV4dGVuZChyZWNvcmQpO29mZnNldHMuYXBwZW5kKGxlbihyYXcpKTttb2Rlc1ttb2RlXSs9MTtleGNlcHRpb25fY291bnQrPWsKICAgIG9mZnNldHM9bnAuYXNhcnJheShvZmZzZXRzLGR0eXBlPSc8dTQnKQogICAgYmFzZWxpbmU9eC5zaXplKmJpdHMvLzg7Y2FuZGlkYXRlPWxlbihyYXcpK29mZnNldHMubmJ5dGVzCiAgICBwYWNrZXQ9eydiaXRzJzpiaXRzLCdjb3VudCc6bGVuKHgpLCdyYXcnOmJ5dGVzKHJhdyksJ29mZnNldHMnOm9mZnNldHMsCiAgICAgICAgICAgICdjb2RlYm9vayc6bnAuYXNhcnJheShjb2RlYm9vayxkdHlwZT1ucC5mbG9hdDMyKSwnbmF0aXZlJzpjYW5kaWRhdGU+PWJhc2VsaW5lLAogICAgICAgICAgICAnb3JpZ2luYWwnOm9yaWdpbmFsLmNvcHkoKSBpZiBjYW5kaWRhdGU+PWJhc2VsaW5lIGVsc2UgTm9uZX0KICAgIHN0YXRzPXsnYmxvY2tfcm1zX2J1ZGdldCc6YnVkZ2V0LCdibG9ja3MnOmxlbih4KSwncmVmZXJlbmNlX2JpdHNfcGVyX3dlaWdodCc6Yml0cywKICAgICAgICAgICAncmVmZXJlbmNlX2NvZGVfYnl0ZXMnOmJhc2VsaW5lLCdlbmNvZGVkX2NhbmRpZGF0ZV9ieXRlcyc6Y2FuZGlkYXRlLAogICAgICAgICAgICdzZWxlY3RlZF9ieXRlcyc6bWluKGNhbmRpZGF0ZSxiYXNlbGluZSksJ3NlbGVjdGVkX2JpdHNfcGVyX3dlaWdodCc6bWluKGNhbmRpZGF0ZSxiYXNlbGluZSkqOC94LnNpemUsCiAgICAgICAgICAgJ3NlbGVjdGVkX25hdGl2ZV9yZWZlcmVuY2UnOmNhbmRpZGF0ZT49YmFzZWxpbmUsJ2NhbmRpZGF0ZV9tb2Rlcyc6eyduYXRpdmUnOm1vZGVzWzBdLCdvbmViaXQnOm1vZGVzWzFdLCdvbmViaXRfd2l0aF9leGNlcHRpb25zJzptb2Rlc1syXX0sCiAgICAgICAgICAgJ2V4Y2VwdGlvbl92YWx1ZXMnOmV4Y2VwdGlvbl9jb3VudCwnc2hhcmVkX25mNF9zY2FsZV9tZXRhZGF0YV9pbmNsdWRlZCc6RmFsc2V9CiAgICByZXR1cm4gcGFja2V0LHN0YXRzCgpkZWYgZGVjb2RlKHBhY2tldCk6CiAgICBpZiBwYWNrZXRbJ25hdGl2ZSddOnJldHVybiByZWZlcmVuY2UocGFja2V0WydvcmlnaW5hbCddLHBhY2tldFsnYml0cyddLHBhY2tldFsnY29kZWJvb2snXSkKICAgIGJpdHM9cGFja2V0WydiaXRzJ107b3V0PW5wLmVtcHR5KChwYWNrZXRbJ2NvdW50J10sQkxPQ0spLGR0eXBlPW5wLmZsb2F0MzIpCiAgICBmb3IgaSBpbiByYW5nZShwYWNrZXRbJ2NvdW50J10pOgogICAgICAgIHJhdz1wYWNrZXRbJ3JhdyddW2ludChwYWNrZXRbJ29mZnNldHMnXVtpXSk6aW50KHBhY2tldFsnb2Zmc2V0cyddW2krMV0pXTttb2RlPXJhd1swXQogICAgICAgIGlmIG1vZGU9PTA6CiAgICAgICAgICAgIG91dFtpXT1wYWNrZXRbJ2NvZGVib29rJ11bdW5uaWJibGVzKHJhd1sxOl0sQkxPQ0spXSBpZiBiaXRzPT00IGVsc2UgbnAuZnJvbWJ1ZmZlcihyYXdbMTpdLGR0eXBlPSc8ZjInKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHNpZ25zPW5wLnVucGFja2JpdHMobnAuZnJvbWJ1ZmZlcihyYXdbMTo5XSxkdHlwZT1ucC51aW50OCksYml0b3JkZXI9J2xpdHRsZScpLmFzdHlwZShib29sKQogICAgICAgIGFscGhhPW5wLmZyb21idWZmZXIocmF3Wzk6MTFdLGR0eXBlPSc8ZjInKVswXQogICAgICAgIG91dFtpXT1ucC53aGVyZShzaWducywxLiwtMS4pLmFzdHlwZShucC5mbG9hdDMyKSpucC5mbG9hdDMyKGFscGhhKQogICAgICAgIGlmIG1vZGU9PTI6CiAgICAgICAgICAgIG1hc2s9bnAudW5wYWNrYml0cyhucC5mcm9tYnVmZmVyKHJhd1sxMToxOV0sZHR5cGU9bnAudWludDgpLGJpdG9yZGVyPSdsaXR0bGUnKS5hc3R5cGUoYm9vbCk7az1pbnQobWFzay5zdW0oKSkKICAgICAgICAgICAgb3V0W2ksbWFza109cGFja2V0Wydjb2RlYm9vayddW3VubmliYmxlcyhyYXdbMTk6XSxrKV0gaWYgYml0cz09NCBlbHNlIG5wLmZyb21idWZmZXIocmF3WzE5Ol0sZHR5cGU9JzxmMicpCiAgICAgICAgZWxpZiBtb2RlIT0xOnJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gcmVjb3JkIG1vZGUnKQogICAgcmV0dXJuIG91dAoKZGVmIGVycm9yX3N1bW1hcnkoYWN0dWFsLGV4cGVjdGVkKToKICAgIGE9bnAuYXNhcnJheShhY3R1YWwsZHR5cGU9bnAuZmxvYXQ2NCk7ZT1ucC5hc2FycmF5KGV4cGVjdGVkLGR0eXBlPW5wLmZsb2F0NjQpCiAgICBkaWZmPWEtZTtkZW49bnAuc3VtKGUqZSxheGlzPTEpO251bT1ucC5zdW0oZGlmZipkaWZmLGF4aXM9MSkKICAgIHJlbD1ucC5zcXJ0KG5wLmRpdmlkZShudW0sZGVuLG91dD1ucC56ZXJvc19saWtlKG51bSksd2hlcmU9ZGVuIT0wKSkKICAgIGlmIG5wLmFueSgoZGVuPT0wKSYobnVtIT0wKSk6cmFpc2UgVmFsdWVFcnJvcignWmVybyByZWZlcmVuY2UgY2hhbmdlZCcpCiAgICByZXR1cm4geydyZWxhdGl2ZV9sMic6ZmxvYXQobnAuc3FydChucC5zdW0obnVtKS9tYXgobnAuc3VtKGRlbiksMWUtMzAwKSkpLAogICAgICAgICAgICAnbWF4X2Jsb2NrX3JlbGF0aXZlX2wyJzpmbG9hdChucC5tYXgocmVsKSksJ21heF9hYnMnOmZsb2F0KG5wLm1heChucC5hYnMoZGlmZikpKX0KCmRlZiBhbmFseXplKHZhbHVlcyxiaXRzLGNvZGVib29rPU5GNCk6CiAgICB4PXJlZmVyZW5jZSh2YWx1ZXMsYml0cyxjb2RlYm9vaykKICAgIGFscGhhPW5wLm1lYW4obnAuYWJzKHgpLGF4aXM9MSxkdHlwZT1ucC5mbG9hdDMyKS5hc3R5cGUobnAuZmxvYXQxNikuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBiaW5hcnk9bnAud2hlcmUoeD49MCwxLiwtMS4pLmFzdHlwZShucC5mbG9hdDMyKSphbHBoYVs6LE5vbmVdCiAgICByZXN1bHQ9eydwdXJlX29uZWJpdCc6eydiaXRzX3Blcl93ZWlnaHQnOjEuMjUsJ3NjYWxlX2JpdHNfcGVyX2Jsb2NrJzoxNiwKICAgICAgICAgICAgJ2Vycm9yJzplcnJvcl9zdW1tYXJ5KGJpbmFyeSx4KX0sJ2FkYXB0aXZlJzpbXX0KICAgIGZvciBidWRnZXQgaW4gKDAuLC4wMSwuMDMsLjEsLjIsLjQsLjYpOgogICAgICAgIHBhY2tldCxzdGF0cz1lbmNvZGUodmFsdWVzLGJpdHMsYnVkZ2V0LGNvZGVib29rKTtkZWNvZGVkPWRlY29kZShwYWNrZXQpCiAgICAgICAgc3RhdHNbJ2Vycm9yJ109ZXJyb3Jfc3VtbWFyeShkZWNvZGVkLHgpCiAgICAgICAgYXNzZXJ0IHN0YXRzWydlcnJvciddWydtYXhfYmxvY2tfcmVsYXRpdmVfbDInXTw9YnVkZ2V0KzFlLTYKICAgICAgICBpZiBidWRnZXQ9PTA6YXNzZXJ0IG5wLmFycmF5X2VxdWFsKGRlY29kZWQseCkKICAgICAgICByZXN1bHRbJ2FkYXB0aXZlJ10uYXBwZW5kKHN0YXRzKQogICAgcmV0dXJuIHJlc3VsdAoKZGVmIHNlbGZfY2hlY2soKToKICAgIGFzc2VydCBuaWJibGVzKG5wLmFyYW5nZSgxNixkdHlwZT1ucC51aW50OCkpPT1ieXRlcy5mcm9taGV4KCcwMTIzNDU2Nzg5YWJjZGVmJykKICAgIGFzc2VydCBucC5hcnJheV9lcXVhbCh1bm5pYmJsZXMoYnl0ZXMuZnJvbWhleCgnMDEyMzQ1Njc4OWFiY2RlZicpLDE2KSxucC5hcmFuZ2UoMTYpKQogICAgcm5nPW5wLnJhbmRvbS5kZWZhdWx0X3JuZygyMDI2MTAzMSkKICAgIGNhc2VzPVsoJ2FsbF9jb2RlcycsbnAudGlsZShucC5hcmFuZ2UoMTYsZHR5cGU9bnAudWludDgpLDEyOCkucmVzaGFwZSgtMSxCTE9DSyksNCksCiAgICAgICAgICAgKCd6ZXJvX25mNCcsbnAuZnVsbCgoOCxCTE9DSyksNyxkdHlwZT1ucC51aW50OCksNCksCiAgICAgICAgICAgKCdiaW5hcnlfZXhhY3RfbmY0JyxybmcuY2hvaWNlKFswLDE1XSwoMzIsQkxPQ0spKS5hc3R5cGUobnAudWludDgpLDQpLAogICAgICAgICAgICgnaGFsZl9yYW5kb20nLHJuZy5ub3JtYWwoMCwuMDIsKDMyLEJMT0NLKSkuYXN0eXBlKG5wLmZsb2F0MTYpLDE2KSwKICAgICAgICAgICAoJ2hhbGZfYmluYXJ5JyxybmcuY2hvaWNlKFstMS4sMS5dLCgzMixCTE9DSykpLmFzdHlwZShucC5mbG9hdDE2KSwxNildCiAgICBmb3IgXyx2YWx1ZXMsYml0cyBpbiBjYXNlczphbmFseXplKHZhbHVlcyxiaXRzKQogICAgcmV0dXJuIHsnY2FzZXMnOlt4WzBdIGZvciB4IGluIGNhc2VzXSwnYnVkZ2V0c19wZXJfY2FzZSc6NywnZ29sZGVuX25pYmJsZV9sYXlvdXQnOlRydWUsCiAgICAgICAgICAgICdyb3VuZHRyaXBfYW5kX2Vycm9yX2JvdW5kX2NoZWNrcyc6MzUsJ3Njb3BlJzonQ1BVX0NPREVDX09OTFlfTk9UX1RSQUlOSU5HX1FVQUxJVFknfQo=', 'gpu_weight_probe.py': 'IiIiUmVhZC1vbmx5IGFjdHVhbC1Rd2VuIHdlaWdodCBzdHVkeTsgbm90IHRyYWluaW5nIG9yIGEgYmVuY2htYXJrIHN1Ym1pc3Npb24uIiIiCmltcG9ydCB0aW1lClNUQVJUPXRpbWUucGVyZl9jb3VudGVyKCkKaW1wb3J0IGdjLGpzb24sb3MsdHJhY2ViYWNrCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCBiaXRzYW5kYnl0ZXMgYXMgYm5iCmZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBzbmFwc2hvdF9kb3dubG9hZApmcm9tIHNhZmV0ZW5zb3JzIGltcG9ydCBzYWZlX29wZW4KZnJvbSBjb2RlYyBpbXBvcnQgYW5hbHl6ZSxzZWxmX2NoZWNrLE5GNCxlcnJvcl9zdW1tYXJ5CgpNT0RFTD0nUXdlbi9Rd2VuMi41LTEuNUItSW5zdHJ1Y3QnClJFVklTSU9OPSc5ODlhYTc5ODBlNGNmODA2ZjgwYzdmZWYyYjFhZGI3YmM3MWFhMzA2JwpPVVQ9UGF0aChvcy5lbnZpcm9uWydTT1VQX1JFU1VMVFMnXSkvJ29uZWJpdF93ZWlnaHRfcHJvYmUuanNvbicKcmVwb3J0PXsnc3RhdHVzJzonUlVOTklORycsJ3Njb3BlJzonQUNUVUFMX1dFSUdIVF9DT01QT05FTlRfUFJPQkVfTk9UX1RSQUlOSU5HX09SX0JFTkNITUFSSycsCiAnbW9kZWwnOk1PREVMLCdyZXZpc2lvbic6UkVWSVNJT04sJ21vZGVsX291dHB1dHNfZXZhbHVhdGVkJzpGYWxzZSwnb2ZmaWNpYWxfYWRtaXNzaW9uJzonVU5LTk9XTicsCiAnb3JpZ2luYWxfa2VybmVsX2NyZWRpdCc6MCwnc2FtcGxlcyc6W10sJ21hdHJpeF9tdWx0aXBseV9wcm9iZXMnOltdLAogJ2xpbWl0YXRpb25zJzpbJ1dlaWdodCBlcnJvciBidWRnZXRzIGFyZSBub3QgdHJhaW5pbmcvZXZhbCBsb3NzIHRvbGVyYW5jZXMnLAogJ05vIGZ1bGwtbW9kZWwgcXVhbGl0eSwgY29tcGxldGVkIDY0LXN0ZXAgdGFzaywgc3BlZWQgb3IgdGFyZ2V0IGNsYWltJywKICdPbmx5IG9uZSBzaWduL3NjYWxlL2V4Y2VwdGlvbiBjb2RlYzsgbm90IGFuIGltcG9zc2liaWxpdHkgcmVzdWx0JywKICdORjQgc2NhbGUgbWV0YWRhdGEgaXMgc2hhcmVkIGFuZCBleGNsdWRlZCBvbiBib3RoIHNpZGVzIG9mIHN0b3JhZ2UgY29tcGFyaXNvbicsCiAnQWxsIHByZXBhcmF0aW9uIGJlbG9uZ3MgdG8gdGhpcyBkaWFnbm9zdGljOyBubyBwcmVjb21wdXRlZCBtb2RlbCB0ZW5zb3JzIGFyZSBleHBvcnRlZCddfQoKZGVmIHNhdmUoKToKICAgIE9VVC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBPVVQud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCxpbmRlbnQ9MikrJ1xuJykKCmRlZiBzZWxlY3RlZF9ncm91cHMoY291bnQpOgogICAgcmV0dXJuIG5wLnVuaXF1ZShucC5saW5zcGFjZSgwLGNvdW50LTEsbWluKGNvdW50LDIwNDgpLGR0eXBlPW5wLmludDY0KSkKCmRlZiBzb3VyY2VfZmlsZShzbmFwc2hvdCxuYW1lKToKICAgIGluZGV4PXNuYXBzaG90Lydtb2RlbC5zYWZldGVuc29ycy5pbmRleC5qc29uJwogICAgaWYgaW5kZXguZXhpc3RzKCk6CiAgICAgICAgZmlsZT1qc29uLmxvYWRzKGluZGV4LnJlYWRfdGV4dCgpKVsnd2VpZ2h0X21hcCddW25hbWVdCiAgICAgICAgaWYgUGF0aChmaWxlKS5uYW1lIT1maWxlOnJhaXNlIFZhbHVlRXJyb3IoJ1VuZXhwZWN0ZWQgc2hhcmQgZmlsZW5hbWUnKQogICAgICAgIHJldHVybiBzbmFwc2hvdC9maWxlCiAgICBmaWxlPXNuYXBzaG90Lydtb2RlbC5zYWZldGVuc29ycycKICAgIGlmIG5vdCBmaWxlLmlzX2ZpbGUoKTpyYWlzZSBSdW50aW1lRXJyb3IoJ1ByZXBhcmVkIHB1YmxpYyBzYWZldGVuc29ycyBtaXNzaW5nJykKICAgIHJldHVybiBmaWxlCgp0cnk6CiAgICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnLCdQaW5uZWQgVG9yY2ggcmVxdWlyZWQnCiAgICBhc3NlcnQgdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKT09MSBhbmQgdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCk9PSdUZXNsYSBUNCcKICAgIHRvcmNoLnNldF9udW1fdGhyZWFkcygyKQogICAgcmVwb3J0WydzZWxmX2NoZWNrJ109c2VsZl9jaGVjaygpO3JlcG9ydFsnZGV2aWNlJ109dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkKICAgIHNuYXBzaG90PVBhdGgoc25hcHNob3RfZG93bmxvYWQoTU9ERUwscmV2aXNpb249UkVWSVNJT04sbG9jYWxfZmlsZXNfb25seT1UcnVlKSkKICAgIGNvbmZpZz1qc29uLmxvYWRzKChzbmFwc2hvdC8nY29uZmlnLmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGFzc2VydCAoY29uZmlnWydoaWRkZW5fc2l6ZSddLGNvbmZpZ1snbnVtX2hpZGRlbl9sYXllcnMnXSxjb25maWdbJ3ZvY2FiX3NpemUnXSk9PSgxNTM2LDI4LDE1MTkzNikKICAgIG5hbWVzPVtmJ21vZGVsLmxheWVycy57aX0ue3N1ZmZpeH0ud2VpZ2h0JyBmb3IgaSBpbiAoMCwxMywyNykKICAgICAgICAgICBmb3Igc3VmZml4IGluICgnc2VsZl9hdHRuLnFfcHJvaicsJ21scC5kb3duX3Byb2onKV0KICAgIG5hbWVzKz1bJ21vZGVsLmxheWVycy4xMy5zZWxmX2F0dG4udl9wcm9qLndlaWdodCcsJ21vZGVsLmxheWVycy4wLm1scC51cF9wcm9qLndlaWdodCddCiAgICBmb3IgbmFtZSBpbiBuYW1lczoKICAgICAgICBpZiB0aW1lLnBlcmZfY291bnRlcigpLVNUQVJUPjE4MDpyYWlzZSBSdW50aW1lRXJyb3IoJ0NvbXBvbmVudCBwcm9iZSB0aW1lIGJ1ZGdldCBleGhhdXN0ZWQnKQogICAgICAgIHdpdGggc2FmZV9vcGVuKHNvdXJjZV9maWxlKHNuYXBzaG90LG5hbWUpLGZyYW1ld29yaz0ncHQnLGRldmljZT0nY3B1JykgYXMgZjoKICAgICAgICAgICAgb3JpZ2luYWw9Zi5nZXRfdGVuc29yKG5hbWUpLnRvKGR0eXBlPXRvcmNoLmZsb2F0MTYpLmNvbnRpZ3VvdXMoKQogICAgICAgIHdlaWdodD1vcmlnaW5hbC50bygnY3VkYScpO2RlbCBvcmlnaW5hbAogICAgICAgIHBhY2tlZCxzdGF0ZT1ibmIuZnVuY3Rpb25hbC5xdWFudGl6ZV80Yml0KHdlaWdodCxibG9ja3NpemU9NjQsY29tcHJlc3Nfc3RhdGlzdGljcz1UcnVlLHF1YW50X3R5cGU9J25mNCcpCiAgICAgICAgbmF0aXZlPWJuYi5mdW5jdGlvbmFsLmRlcXVhbnRpemVfNGJpdChwYWNrZWQscXVhbnRfc3RhdGU9c3RhdGUpCiAgICAgICAgaW5kaWNlcz1zZWxlY3RlZF9ncm91cHMobmF0aXZlLm51bWVsKCkvLzY0KTtpbmRleD10b3JjaC50ZW5zb3IoaW5kaWNlcyxkZXZpY2U9J2N1ZGEnKQogICAgICAgIHJhdz1wYWNrZWQucmVzaGFwZSgtMSwzMikuaW5kZXhfc2VsZWN0KDAsaW5kZXgpLmNwdSgpLm51bXB5KCkKICAgICAgICBjb2Rlcz1ucC5lbXB0eSgobGVuKGluZGljZXMpLDY0KSxkdHlwZT1ucC51aW50OCk7Y29kZXNbOiw6OjJdPXJhdz4+NDtjb2Rlc1s6LDE6OjJdPXJhdyYxNQogICAgICAgIHNjYWxlcz1ibmIuZnVuY3Rpb25hbC5kZXF1YW50aXplX2Jsb2Nrd2lzZShzdGF0ZS5hYnNtYXgscXVhbnRfc3RhdGU9c3RhdGUuc3RhdGUyKStzdGF0ZS5vZmZzZXQKICAgICAgICBjaG9zZW5fc2NhbGVzPXNjYWxlcy5pbmRleF9zZWxlY3QoMCxpbmRleCkuY3B1KCkubnVtcHkoKQogICAgICAgIHNlbGVjdGVkX25hdGl2ZT1uYXRpdmUucmVzaGFwZSgtMSw2NCkuaW5kZXhfc2VsZWN0KDAsaW5kZXgpLmNwdSgpLm51bXB5KCkKICAgICAgICBjb2RlYm9vaz1zdGF0ZS5jb2RlLmNwdSgpLm51bXB5KCkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgcmVjb25zdHJ1Y3RlZD0oY29kZWJvb2tbY29kZXNdKmNob3Nlbl9zY2FsZXNbOixOb25lXSkuYXN0eXBlKG5wLmZsb2F0MTYpCiAgICAgICAgYXNzZXJ0IG5wLmFycmF5X2VxdWFsKHJlY29uc3RydWN0ZWQudmlldyhucC51aW50MTYpLHNlbGVjdGVkX25hdGl2ZS52aWV3KG5wLnVpbnQxNikpLCAnTkY0IHJlZmVyZW5jZSByZWNvbnN0cnVjdGlvbiBtaXNtYXRjaCcKICAgICAgICBzdGFydGVkPXRpbWUucGVyZl9jb3VudGVyKCk7YW5hbHlzaXM9YW5hbHl6ZShjb2Rlcyw0LGNvZGVib29rKQogICAgICAgIHJlcG9ydFsnc2FtcGxlcyddLmFwcGVuZCh7J3RlbnNvcic6bmFtZSwnc2hhcGUnOmxpc3Qod2VpZ2h0LnNoYXBlKSwnc2FtcGxlZF9ibG9ja3MnOmxlbihpbmRpY2VzKSwKICAgICAgICAgICdzYW1wbGVfc2VsZWN0aW9uJzonRXZlbmx5IHNwYWNlZCBhbGlnbmVkIGJsb2NrcyBhY3Jvc3MgdGhlIGNvbXBsZXRlIHRlbnNvcicsCiAgICAgICAgICAncmVmZXJlbmNlJzonTmF0aXZlIGJpdHNhbmRieXRlcyB3aG9sZS10ZW5zb3IgTkY0IHdpdGggZG91YmxlIHF1YW50aXphdGlvbiwgZGVjb2RlZCBGUDE2JywKICAgICAgICAgICdyZWZlcmVuY2VfcmVjb25zdHJ1Y3Rpb25fYml0X2V4YWN0JzpUcnVlLCdjb2RlY19wcmVwYXJhdGlvbl9hbmRfYW5hbHlzaXNfcyc6dGltZS5wZXJmX2NvdW50ZXIoKS1zdGFydGVkLAogICAgICAgICAgJ2FuYWx5c2lzJzphbmFseXNpc30pCiAgICAgICAgaWYgbmFtZSBpbiAobmFtZXNbMF0sbmFtZXNbM10pOgogICAgICAgICAgICB0b3JjaC5tYW51YWxfc2VlZCgxNykKICAgICAgICAgICAgeD10b3JjaC5yYW5kbigoMzIsbmF0aXZlLnNoYXBlWzFdKSxkZXZpY2U9J2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MTYpCiAgICAgICAgICAgIGJsb2Nrcz1uYXRpdmUucmVzaGFwZSgtMSw2NCkuZmxvYXQoKQogICAgICAgICAgICBhbHBoYT1ibG9ja3MuYWJzKCkubWVhbihkaW09MSxrZWVwZGltPVRydWUpLmhhbGYoKQogICAgICAgICAgICBiaW5hcnk9KHRvcmNoLndoZXJlKGJsb2Nrcz49MCwxLiwtMS4pKmFscGhhKS5oYWxmKCkucmVzaGFwZV9hcyhuYXRpdmUpCiAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICAgICAgYmFzZT10b3JjaC5ubi5mdW5jdGlvbmFsLmxpbmVhcih4LG5hdGl2ZSkKICAgICAgICAgICAgICAgIGNoYW5nZWQ9dG9yY2gubm4uZnVuY3Rpb25hbC5saW5lYXIoeCxiaW5hcnkpCiAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgICAgICAgICByZXBvcnRbJ21hdHJpeF9tdWx0aXBseV9wcm9iZXMnXS5hcHBlbmQoeyd0ZW5zb3InOm5hbWUsJ2lucHV0cyc6J1N5bnRoZXRpYyBub3JtYWwgRlAxNiwgbm90IHRhc2sgYWN0aXZhdGlvbnMnLAogICAgICAgICAgICAgICdlcnJvcic6ZXJyb3Jfc3VtbWFyeShjaGFuZ2VkLmZsb2F0KCkuY3B1KCkubnVtcHkoKSxiYXNlLmZsb2F0KCkuY3B1KCkubnVtcHkoKSksCiAgICAgICAgICAgICAgJ3Njb3BlJzonUHJvamVjdGlvbiBzZW5zaXRpdml0eSBvbmx5OyBubyBzcGVlZCBvciB0YXNrLXF1YWxpdHkgY2xhaW0nfSkKICAgICAgICAgICAgZGVsIHgsYmxvY2tzLGFscGhhLGJpbmFyeSxiYXNlLGNoYW5nZWQKICAgICAgICBkZWwgd2VpZ2h0LHBhY2tlZCxzdGF0ZSxuYXRpdmUsaW5kZXgsc2NhbGVzCiAgICAgICAgZ2MuY29sbGVjdCgpO3NhdmUoKQogICAgbmFtZT0nbW9kZWwuZW1iZWRfdG9rZW5zLndlaWdodCcKICAgIHdpdGggc2FmZV9vcGVuKHNvdXJjZV9maWxlKHNuYXBzaG90LG5hbWUpLGZyYW1ld29yaz0ncHQnLGRldmljZT0nY3B1JykgYXMgZjoKICAgICAgICB2aWV3PWYuZ2V0X3NsaWNlKG5hbWUpO3NoYXBlPXZpZXcuZ2V0X3NoYXBlKCkKICAgICAgICByb3dzPVswLHNoYXBlWzBdLy8yLHNoYXBlWzBdLTMyXQogICAgICAgIGVtYmVkZGluZz10b3JjaC5jYXQoW3ZpZXdbcjpyKzMyLDpdLnRvKGR0eXBlPXRvcmNoLmZsb2F0MTYpIGZvciByIGluIHJvd3NdKS5udW1weSgpLnJlc2hhcGUoLTEsNjQpCiAgICBpbmRpY2VzPXNlbGVjdGVkX2dyb3VwcyhsZW4oZW1iZWRkaW5nKSk7ZW1iZWRkaW5nPWVtYmVkZGluZ1tpbmRpY2VzXQogICAgc3RhcnRlZD10aW1lLnBlcmZfY291bnRlcigpCiAgICByZXBvcnRbJ3NhbXBsZXMnXS5hcHBlbmQoeyd0ZW5zb3InOm5hbWUsJ3NoYXBlJzpzaGFwZSwnc2FtcGxlZF9ibG9ja3MnOmxlbihpbmRpY2VzKSwKICAgICAgICdzYW1wbGVfc2VsZWN0aW9uJzonRXZlbmx5IHNwYWNlZCBibG9ja3MgZnJvbSB0aHJlZSAzMi1yb3cgd2luZG93czogZmlyc3QsIG1pZGRsZSwgbGFzdCcsCiAgICAgICAncmVmZXJlbmNlJzonT3JpZ2luYWwgcHVibGljIGNoZWNrcG9pbnQgdmFsdWVzIHJldGFpbmVkIGluIEZQMTY7IG5vIGV4dHJhIE5GNCBxdWFudGl6YXRpb24nLAogICAgICAgJ2FuYWx5c2lzJzphbmFseXplKGVtYmVkZGluZywxNiksJ2NvZGVjX3ByZXBhcmF0aW9uX2FuZF9hbmFseXNpc19zJzp0aW1lLnBlcmZfY291bnRlcigpLXN0YXJ0ZWR9KQogICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpO3JlcG9ydFsnc3RhdHVzJ109J0NPTVBMRVRFX0NPTVBPTkVOVF9TVFVEWScKZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgcmVwb3J0WydzdGF0dXMnXT0nRkFJTEVEX0NPTVBPTkVOVF9TVFVEWSc7cmVwb3J0WydlcnJvciddPXJlcHIoZXhjKQogICAgcmVwb3J0Wyd0cmFjZWJhY2snXT10cmFjZWJhY2suZm9ybWF0X2V4YygpCmZpbmFsbHk6CiAgICByZXBvcnRbJ2RpYWdub3N0aWNfd2FsbF9zJ109dGltZS5wZXJmX2NvdW50ZXIoKS1TVEFSVDtzYXZlKCkKcHJpbnQoanNvbi5kdW1wcyh7J3N0YXR1cyc6cmVwb3J0WydzdGF0dXMnXSwnc2FtcGxlcyc6bGVuKHJlcG9ydFsnc2FtcGxlcyddKSwKICAgICAgICAgICAgICAgICAgJ2RpYWdub3N0aWNfd2FsbF9zJzpyZXBvcnRbJ2RpYWdub3N0aWNfd2FsbF9zJ10sJ25vdF9hX2JlbmNobWFyayc6VHJ1ZX0pKQppZiByZXBvcnRbJ3N0YXR1cyddIT0nQ09NUExFVEVfQ09NUE9ORU5UX1NUVURZJzpyYWlzZSBSdW50aW1lRXJyb3IocmVwb3J0LmdldCgnZXJyb3InLCdJbmNvbXBsZXRlIGNvbXBvbmVudCBwcm9iZScpKQo='}
HASHES={'codec.py': '5f348a13aa9aa8c16c81d3382145807056a9d9030b075101e8cac1a41a2e78b4', 'gpu_weight_probe.py': '8ab71a3859ceca2844aec54a32faac0218a11c49ac2f20e01ca456a14ee7e5a2'}
stage=Path(tempfile.mkdtemp(prefix='onebit-weight-probe-',dir='/kaggle/working'))
for name,value in PAYLOAD.items():
 raw=base64.b64decode(value,validate=True)
 assert hashlib.sha256(raw).hexdigest()==HASHES[name]
 (stage/name).write_bytes(raw)
sys.path.insert(0,str(stage))
runpy.run_path(str(stage/'gpu_weight_probe.py'),run_name='__main__')
